In [3]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import time
import warnings
warnings.filterwarnings('ignore')

print("Library berhasil di-load!")

Library berhasil di-load!


## Dataset Reality Check (Standar Gaji)

In [4]:
reality_check_data = {
    "country": ["South Korea", "South Korea", "Japan", "Japan", "Taiwan", "Taiwan", "Malaysia", "Saudi Arabia"],
    "job_sector": ["Manufacturing", "Agriculture", "Caregiver", "Manufacturing", "Caregiver", "Manufacturing", "Domestic Worker", "Domestic Worker"],
    "est_annual_salary_usd": [18500, 18000, 16000, 17500, 10500, 11500, 4500, 5000],
    "max_realistic_salary_usd": [25000, 23000, 22000, 24000, 15000, 17000, 8000, 9000] 
}

df_reality = pd.DataFrame(reality_check_data)
display(df_reality.head())

,country,job_sector,est_annual_salary_usd,max_realistic_salary_usd
0,South Korea,Manufacturing,18500,25000
1,South Korea,Agriculture,18000,23000
2,Japan,Caregiver,16000,22000
3,Japan,Manufacturing,17500,24000
4,Taiwan,Caregiver,10500,15000


## Dataset Geo Risk & OCIndex (Organized Crime Index)

In [5]:
ocindex_data = {
    "country": ["South Korea", "Japan", "Taiwan", "Malaysia", "Saudi Arabia", "Syria", "Myanmar"],
    "human_trafficking_score": [3.5, 4.0, 3.0, 6.5, 7.0, 8.5, 9.0],
    "resilience_score": [7.5, 8.0, 7.5, 5.0, 4.5, 1.5, 2.0],
}

df_ocindex = pd.DataFrame(ocindex_data)

df_ocindex['geo_risk_index'] = df_ocindex['human_trafficking_score'] / df_ocindex['resilience_score']

df_ocindex['geo_risk_score'] = pd.cut(
    df_ocindex['geo_risk_index'], 
    bins=[0, 0.7, 1.5, 10], 
    labels=[1, 2, 3] # 1: Aman, 2: Waspada, 3: Bahaya (Zona Merah)
).astype(int)

display(df_ocindex)

,country,human_trafficking_score,resilience_score,geo_risk_index,geo_risk_score
0,South Korea,3.5,7.5,0.466667,1
1,Japan,4.0,8.0,0.500000,1
2,Taiwan,3.0,7.5,0.400000,1
3,Malaysia,6.5,5.0,1.300000,2
4,Saudi Arabia,7.0,4.5,1.555556,3
5,Syria,8.5,1.5,5.666667,3
6,Myanmar,9.0,2.0,4.500000,3


## Scraper BP2MI (Untuk Cek Entitas Loker)

In [6]:
def scrape_bp2mi_jobs():
    url = "https://siskop2mi.bp2mi.go.id/lowongan/list"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    jobs_data = []

    print(f"Mengakses URL: {url}...")
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        job_cards = soup.find_all('div', class_='card') 
        print(f"Ditemukan {len(job_cards)} elemen di halaman pertama.")
        
        target_countries = ['korea', 'jepang', 'japan', 'taiwan']
        
        for card in job_cards:
            try:
                title = card.find('h5').text.strip() if card.find('h5') else 'N/A'
                country = "Unknown"
                
                details = card.find_all('li')
                for detail in details:
                    if 'negara' in detail.text.lower():
                        country = detail.text.split(':')[-1].strip()
                
                if any(tc in country.lower() for tc in target_countries):
                    jobs_data.append({'job_title': title, 'country': country, 'is_official_bp2mi': 1})
            except Exception:
                continue
                
    except Exception as e:
        print(f"Scraping gagal/diblokir: {e}")

    if not jobs_data:
        print("Data tidak ditemukan atau web memblokir scraping. Menggunakan data fallback...")
        jobs_data = [
            {'job_title': 'Pekerja Manufaktur', 'country': 'South Korea', 'is_official_bp2mi': 1},
            {'job_title': 'Caregiver Lansia', 'country': 'Japan', 'is_official_bp2mi': 1}
        ]

    return pd.DataFrame(jobs_data)

df_bp2mi = scrape_bp2mi_jobs()
display(df_bp2mi.head())

Mengakses URL: https://siskop2mi.bp2mi.go.id/lowongan/list...
Ditemukan 0 elemen di halaman pertama.
Data tidak ditemukan atau web memblokir scraping. Menggunakan data fallback...


,job_title,country,is_official_bp2mi
0,Pekerja Manufaktur,South Korea,1
1,Caregiver Lansia,Japan,1


## MENGGABUNGKAN FITUR EKSTERNAL KE DATASET UTAMA

In [7]:
print("Menyimulasikan dataset lowongan (Main Data)...")

df_main = pd.DataFrame({
    'job_id': [101, 102, 103, 104],
    'job_title': ['Pekerja Manufaktur', 'IT Support', 'Caregiver', 'Asisten Rumah Tangga'],
    'country': ['South Korea', 'Taiwan', 'Japan', 'Malaysia'],
    'job_sector': ['Manufacturing', 'IT', 'Caregiver', 'Domestic Worker'],
    'salary_offered_usd': [26000, 12000, 30000, 10000]
})

# 1. Merge dengan Geo Risk (OCIndex)
df_main = df_main.merge(df_ocindex[['country', 'human_trafficking_score', 'geo_risk_score']], on='country', how='left')

# 2. Merge dengan Reality Check (Gaji Wajar)
df_main = df_main.merge(df_reality[['country', 'job_sector', 'max_realistic_salary_usd']], on=['country', 'job_sector'], how='left')

# 3. Hitung Reality Check Flag
df_main['is_unrealistic_salary'] = np.where(
    df_main['salary_offered_usd'] > df_main['max_realistic_salary_usd'], 1, 0
)

# 4. Cek apakah loker terdaftar resmi di BP2MI
df_main = df_main.merge(df_bp2mi[['job_title', 'country', 'is_official_bp2mi']], on=['job_title', 'country'], how='left')
df_main['is_official_bp2mi'] = df_main['is_official_bp2mi'].fillna(0).astype(int)

print("\n=== HASIL AKHIR PENGGABUNGAN (SIAP MASUK MODEL XGBOOST) ===")
display(df_main)

Menyimulasikan dataset lowongan (Main Data)...

=== HASIL AKHIR PENGGABUNGAN (SIAP MASUK MODEL XGBOOST) ===


,job_id,job_title,country,job_sector,salary_offered_usd,human_trafficking_score,geo_risk_score,max_realistic_salary_usd,is_unrealistic_salary,is_official_bp2mi
0,101,Pekerja Manufaktur,South Korea,Manufacturing,26000,3.5,1,25000.0,1,1
1,102,IT Support,Taiwan,IT,12000,3.0,1,NaN,0,0
2,103,Caregiver,Japan,Caregiver,30000,4.0,1,22000.0,1,0
3,104,Asisten Rumah Tangga,Malaysia,Domestic Worker,10000,6.5,2,8000.0,1,0
